# Fase 2.1: Limpieza de datos

Tratamos los registros inválidos detectados en el EDA, los valores faltantes y los outliers extremos, documentando cada decisión.

Entrada `data/processed/dataset_unificado_0.csv` (versión 2021-2023 elegida en el EDA, una fila por día con el último registro de las 23:59).

No se guarda ningún archivo final en este notebook. Solo se evalúan y aplican las reglas de limpieza.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42
carpeta_datos = Path("..") / "data"

df = pd.read_csv(carpeta_datos / "processed" / "dataset_unificado_0.csv", parse_dates=["dia"])
df = df.set_index("dia")
df.shape

(1213, 293)

## 1. Verificación del calendario

Cada fila tiene que corresponder a un día distinto y no puede haber días faltantes dentro del rango.

In [2]:
calendario = pd.date_range(df.index.min(), df.index.max(), freq="D")
dias_faltantes = calendario.difference(df.index)
print("Días en el rango:", len(calendario))
print("Filas en el dataset:", len(df))
print("Días faltantes en el calendario:", len(dias_faltantes))
print("Fechas duplicadas en el índice:", df.index.duplicated().sum())

Días en el rango: 1213
Filas en el dataset: 1213
Días faltantes en el calendario: 0
Fechas duplicadas en el índice: 0


## 2. Días incompletos



In [3]:
# df.isna() marca con True cada celda vacía.
# .sum(axis=1) suma por fila: para cada día, cuántas columnas están vacías.
columnas_vacias = df.isna().sum(axis=1)

# La misma información como fracción del total de columnas (0 = completo, 1 = todo vacío)
nulos_por_dia = df.isna().mean(axis=1)

columnas_vacias.describe()

count    1213.000000
mean       15.890354
std        23.518906
min         1.000000
25%         7.000000
50%        12.000000
75%        24.000000
max       293.000000
dtype: float64

In [4]:
top = columnas_vacias.sort_values(ascending=False).head(20).to_frame("columnas_vacias")
top["frio_kw"] = df.loc[top.index, "frio_kw"]
top

,columnas_vacias,frio_kw
dia,,
2023-10-26,293,NaN
2021-07-13,292,NaN
2023-04-19,292,NaN
2023-04-13,292,NaN
2022-03-02,292,NaN
2023-08-30,292,NaN
2023-02-28,286,NaN
2023-04-16,61,14738.0
2023-06-25,61,15073.0


El 2023-10-26 tiene las 293 vacías. Es el último día del dataset. Le falta hasta frio_kw_siguiente, que es lógico porque no hay día siguiente. Lo eliminamos.

Los días con 292 tienen una sola columna con dato. Veamos si es frio_kw_siguiente:

In [5]:
df = df.drop(index=pd.Timestamp("2023-10-26"))

In [6]:
dias_292 = columnas_vacias[columnas_vacias == 292].index
df.loc[dias_292].notna().sum()[lambda s: s > 0]

frio_kw_siguiente    5
dtype: int64

In [7]:
fila = df.loc["2023-02-28"]
fila[fila.notna()]

totalizador_l2_barriles     1513.00
totalizador_l3              1322.00
totaliador_latas            5796.00
totalizador_bodega          4483.38
totalizador_tq_pulmon      15386.00
totalizador_filtracion       892.00
frio_kw_siguiente          23447.00
Name: 2023-02-28 00:00:00, dtype: float64

Encontramos 5 días que tienen vacías 292 de sus 293 columnas. La única columna con dato es `frio_kw_siguiente`, que no es información de ese día sino el consumo del día siguiente, traído con el `shift(-1)` que se hizo en el EDA. En sus propios datos, estos días están completamente vacíos: corresponden a días sin registro de las 23:59, que en el EDA quedaron como faltantes.

Decidimos **no eliminar estas filas**, por dos motivos:

- Para luego construir variables de lag y promedios móviles con `shift`, que toman la fila anterior de la tabla. Si eliminamos una fila del medio, `shift(1)` deja de ser el día anterior y pasa a ser dos días antes, sin que se note en el código. Mantener todas las fechas del calendario garantiza que cada desplazamiento corresponda a un día real.
- Estos días no tienen ningún dato propio, así que imputarlos sería inventar las 293 columnas de un día del que no sabemos nada.

En su lugar, los marcamos con la columna `dia_vacio` y los **excluimos del entrenamiento**. Por la misma razón, también se excluyen los días anteriores a cada uno de ellos, cuyo `frio_kw_siguiente` está vacío porque el día siguiente no tiene dato.

La única fila que eliminamos es la del 2023-10-26: es el último día del dataset, está completamente vacía (incluso `frio_kw_siguiente`, porque no hay día siguiente), y al estar en el extremo no rompe la continuidad del calendario.

In [8]:
dias_totalmente_vacios = columnas_vacias[columnas_vacias == df.shape[1]]

print("Días totalmente vacíos:", len(dias_totalmente_vacios))
dias_totalmente_vacios.index

Días totalmente vacíos: 1


DatetimeIndex(['2023-10-26'], dtype='datetime64[us]', name='dia', freq=None)

In [9]:
# Elegí un día de la tabla de la celda 3 que no esté totalmente vacío
dia = top.index[1]

faltantes_del_dia = df.loc[dia][df.loc[dia].isna()].index.tolist()
print(dia.date(), "-", len(faltantes_del_dia), "columnas vacías")
faltantes_del_dia

2021-07-13 - 292 columnas vacías


['ee_planta_hl',
 'ee_elaboracion_hl',
 'ee_bodega_hl',
 'ee_cocina_hl',
 'ee_envasado_hl',
 'ee_linea_2_hl',
 'ee_linea_3_hl',
 'ee_linea_4_hl',
 'ee_servicios_hl',
 'ee_sala_maq_hl',
 'ee_frio_hl',
 'ee_aire_hl',
 'ee_co2_hl',
 'ee_caldera_hl',
 'ee_eflu_hl',
 'ee_agua_hl',
 'ee_resto_serv_hl',
 'ee_resto_planta_hl',
 'agua_planta_hl',
 'agua_elab_hl',
 'agua_bodega_hl',
 'agua_cocina_hl',
 'agua_envas_hl',
 'agua_linea_2_hl',
 'agua_linea_3_hl',
 'agua_linea_4_hl',
 'agua_linea_5_hl',
 'agua_servicios_hl',
 'agua_planta_de_agua_hl',
 'produccion_agua_hl',
 'et_planta_hl',
 'et_elab_hl',
 'et_bodega_hl',
 'et_cocina_hl',
 'et_envasado_hl',
 'et_linea_2_hl',
 'et_linea_3_hl',
 'et_linea_4_hl',
 'et_linea_5_hl',
 'et_servicios_hl',
 'aire_planta_hl',
 'aire_elaboracion_hl',
 'aire_cocina_hl',
 'aire_bodega_hl',
 'aire_envasado_hl',
 'aire_l2_hl',
 'aire_l3_hl',
 'aire_l4_hl',
 'aire_l5_hl',
 'aire_servicios_hl',
 'aire_expulsion_hl',
 'co_2_hl',
 'co_2_filtro_hl',
 'co_2_linea_3_hl',
 

### Días que llegan hasta las 23:00

En el EDA nos quedamos con el registro de las 23:59 y dejamos vacíos los días que no lo tienen. Pero algunos de esos días sí tienen lecturas hasta las 23:00, así que no hace falta perderlos: 2021-07-13, 2022-03-02 y 2023-02-28. Los primeros dos están entre los días de 292 columnas vacías que vimos arriba.

Los otros días sin `frio_kw` no entran en este caso: 2023-04-13 llega hasta las 19:00, 2023-04-19 hasta las 16:00, 2023-08-30 no tiene ninguna lectura y 2023-10-26 ya lo eliminamos.

Como el CSV tiene una sola fila por día, volvemos a leer las lecturas horarias del archivo 2021-2023, que es el que tiene prioridad en el dataset unificado. Algunas fechas tienen varios días apilados (ver "Días pisados" en el EDA), y en esos casos el primero es el que le corresponde a esa fecha.

In [23]:
import sys
sys.path.append(str(Path("..") / "src"))
from eda_1_1_auxiliar import a_snake_case, snake_case_unicas, limpiar_numericas

DIAS_HASTA_LAS_23 = pd.to_datetime(["2021-07-13", "2022-03-02", "2023-02-28"])
H_2359 = pd.Timedelta("23:59:00")
H_23 = pd.Timedelta("23:00:00")

hojas = pd.read_excel(carpeta_datos / "Totalizadores Planta - 2021_2023.xlsx", sheet_name=None, engine="calamine")

horarias = {}
for hoja, h in hojas.items():
    h.columns = snake_case_unicas(list(h.columns))
    if not {"dia", "hora"}.issubset(h.columns):
        continue
    h, _ = limpiar_numericas(h)
    h["dia"] = pd.to_datetime(h["dia"], errors="coerce").dt.normalize()
    # Los segundos varían (05:00:01), por eso se redondea al minuto
    h["t"] = pd.to_timedelta(h["hora"].astype(str), errors="coerce").dt.floor("min")
    h = h.dropna(subset=["dia", "t"])
    # Si la hora vuelve a bajar empieza otro día apilado: nos quedamos con el primero de cada fecha
    bloque = (h.groupby("dia")["t"].diff() < pd.Timedelta(0)).groupby(h["dia"]).cumsum()
    horarias[a_snake_case(hoja)] = h[bloque == 0]

ee = horarias["consolidado_ee"]
ee[ee["dia"].isin(DIAS_HASTA_LAS_23) & (ee["t"] >= pd.Timedelta("22:00:00"))][["dia", "hora", "frio_kw"]]

,dia,hora,frio_kw
319,2021-07-13,22:00:00,32204.0
320,2021-07-13,23:00:00,33704.0
6115,2022-03-02,22:00:00,18216.0
6116,2022-03-02,23:00:00,19294.0
15105,2023-02-28,22:00:00,14494.0
15106,2023-02-28,23:00:00,15279.0


Tenemos tres opciones: dejar el día vacío, usar el valor de las 23:00 tal cual, o estimar cuánto se habría sumado en los 59 minutos que faltan. Para elegir, las probamos en los días en los que **sí** conocemos el valor de las 23:59 y medimos cuánto se equivoca cada una.

In [24]:
def foto(h, hora, columna):
    """Valor de una columna a una hora dada, con una fila por día."""
    return h[h["t"] == hora].drop_duplicates("dia").set_index("dia")[columna]

def horas_antes(h, columna):
    """Tabla por día con las lecturas de las 17, 20, 22, 23 y 23:59."""
    return pd.DataFrame({
        "h17": foto(h, pd.Timedelta("17:00:00"), columna),
        "h20": foto(h, pd.Timedelta("20:00:00"), columna),
        "h22": foto(h, pd.Timedelta("22:00:00"), columna),
        "h23": foto(h, H_23, columna),
        "h2359": foto(h, H_2359, columna),
    })

# Se sacan los totales imposibles para que no distorsionen la comparación
p = horas_antes(ee, "frio_kw").dropna()
p = p[(p["h2359"] < 1e6) & (p["h2359"] >= p["h23"])]

frio = df["frio_kw"].mask(df["frio_kw"] >= 1e6)
estimaciones = {
    "Valor de las 23:00 sin corregir": p["h23"],
    "Ritmo de la última hora": p["h23"] + (p["h23"] - p["h22"]) * 59 / 60,
    "Ritmo de las últimas 3 horas": p["h23"] + (p["h23"] - p["h20"]) / 3 * 59 / 60,
    "Ritmo de las últimas 6 horas": p["h23"] + (p["h23"] - p["h17"]) / 6 * 59 / 60,
    "Ritmo promedio del día": p["h23"] * (1 + (59 / 60) / 23),
    "Copiar el día anterior": frio.shift(1).reindex(p.index),
    "Promedio de D-1 y D+1 (usa el futuro)": ((frio.shift(1) + frio.shift(-1)) / 2).reindex(p.index),
}

comparacion = pd.DataFrame({
    nombre: {
        "MAE_kw": (e - p["h2359"]).abs().mean(),
        "error_mediano_kw": (e - p["h2359"]).abs().median(),
        "sesgo_kw": (e - p["h2359"]).mean(),
        "error_p95_kw": (e - p["h2359"]).abs().quantile(0.95),
    }
    for nombre, e in estimaciones.items()
}).T

print("Días usados en la comparación:", len(p))
print("Diferencia media entre 23:59 y 23:00:", round((p["h2359"] - p["h23"]).mean()), "kW")
comparacion.round(1)

Días usados en la comparación: 598
Diferencia media entre 23:59 y 23:00: 975 kW


,MAE_kw,error_mediano_kw,sesgo_kw,error_p95_kw
Valor de las 23:00 sin corregir,975.5,924.5,-975.5,1681.0
Ritmo de la última hora,234.2,136.1,-36.7,581.2
Ritmo de las últimas 3 horas,231.8,128.9,2.4,541.7
Ritmo de las últimas 6 horas,222.8,122.2,7.2,510.3
Ritmo promedio del día,225.4,135.6,-29.9,510.7
Copiar el día anterior,3690.0,2911.0,2.0,9923.6
Promedio de D-1 y D+1 (usa el futuro),2844.4,2120.8,5.2,8260.6


Para elegir el método, usamos días en los que conocemos el valor real de las 23:59: hacemos de cuenta que no lo tenemos, lo estimamos de cada forma y medimos el error contra el valor real. `MAE_kw` es el error promedio, `error_mediano_kw` el error de un día típico, `sesgo_kw` indica si el método tiende a quedarse corto (negativo) o a pasarse (positivo), y `error_p95_kw` es el error que no se supera en el 95% de los días. Extrapolar con el ritmo de las últimas horas da errores de unos 230 kW, mientras que usar el valor de las 23:00 sin corregir se queda corto por unos 975 kW y copiar días vecinos se equivoca en unos 3.000 kW.

`frio_kw` es un totalizador, pero el dataset también tiene columnas que no acumulan, como KPI por hectolitro, niveles y temperaturas. A esas no les podemos sumar lo que falta, porque no crecen durante el día. Consideramos acumuladoras a las que no bajan entre las 23:00 y las 23:59 en al menos el 95% de los días, y al resto les dejamos el valor de las 23:00.

También controlamos que las lecturas coincidan con el CSV: si leemos las 23:59 de los demás días, tenemos que obtener los mismos valores.

In [25]:
UMBRAL_ACUMULADORA = 0.95

columnas_vistas = set()
estimados = {}      # columna -> valores estimados de los días que llegan hasta las 23:00
filas_control = []

for hoja, h in horarias.items():
    for columna in h.columns.drop(["dia", "hora", "t"]):
        # Al unir las hojas, el EDA renombró las columnas repetidas a columna_hoja
        nombre = f"{columna}_{hoja}" if columna in columnas_vistas else columna
        columnas_vistas.add(columna)
        if nombre not in df.columns or not pd.api.types.is_numeric_dtype(h[columna]):
            continue

        horas = horas_antes(h, columna)
        completos = horas.dropna(subset=["h23", "h2359"])
        acumuladora = len(completos) > 0 and (completos["h2359"] >= completos["h23"]).mean() >= UMBRAL_ACUMULADORA

        def estimar(t):
            if not acumuladora:
                return t["h23"]
            # Si no hay lectura a las 20:00 o el totalizador bajó, no se suma nada
            ritmo = ((t["h23"] - t["h20"]) / 3).clip(lower=0).fillna(0)
            return t["h23"] + ritmo * 59 / 60

        estimados[nombre] = estimar(horas).reindex(DIAS_HASTA_LAS_23)

        # Controles sobre los días con 23:59 real
        comunes = completos.index.intersection(df.index)
        if len(comunes) > 0 and completos["h2359"].abs().median() > 0:
            escala = completos["h2359"].abs().median()
            filas_control.append({
                "columna": nombre,
                "acumuladora": acumuladora,
                "coincide_con_csv": np.isclose(completos.loc[comunes, "h2359"], df.loc[comunes, nombre], equal_nan=True).mean(),
                "error_23": (completos["h23"] - completos["h2359"]).abs().median() / escala,
                "error_estimado": (estimar(completos) - completos["h2359"]).abs().median() / escala,
            })

control = pd.DataFrame(filas_control).set_index("columna")
print("Columnas del dataset con lecturas horarias:", len(estimados), "de", df.shape[1])
print("Acumuladoras:", int(control["acumuladora"].sum()), "| No acumuladoras:", int((~control["acumuladora"]).sum()))
print("Coincidencia mediana con el CSV:", round(control["coincide_con_csv"].median(), 4))
control.groupby("acumuladora")[["error_23", "error_estimado"]].median().round(4)

Columnas del dataset con lecturas horarias: 289 de 295
Acumuladoras: 165 | No acumuladoras: 65
Coincidencia mediana con el CSV: 1.0


,error_23,error_estimado
acumuladora,,
False,0.0135,0.0135
True,0.0398,0.0037


**Decisión: estimamos los tres días con el ritmo de las últimas 3 horas.**

Dejarlos vacíos pierde tres días de target y corta los lags que dependen de ellos. Usar el valor de las 23:00 sin corregir queda corrido hacia abajo y, además, mezcla las dos definiciones de total diario que se pide no mezclar: seguimos usando las 23:59, y esto es una estimación de ese valor. Copiar días vecinos es mucho peor, y el promedio con D+1 usa información del futuro.

Entre las opciones de extrapolar las diferencias son chicas: las últimas 6 horas tienen un MAE unos 9 kW menor, pero eso es irrelevante frente a un total diario de más de 20.000 kW. Elegimos las últimas 3 horas porque tienen el menor sesgo, usan el ritmo más cercano al final del día y no dependen de una sola lectura. Además, no necesitan ningún parámetro aprendido de los datos, así que no hay riesgo de filtrar información del test. Su MAE de unos 230 kW queda muy por debajo del límite de 3.000 kW que pide la consigna.

Solo completamos las celdas vacías: el 2023-02-28 tiene 6 columnas con dato real de las 23:59 y esas no se tocan. Los días quedan marcados en `dia_estimado_23h` para poder analizarlos o excluirlos más adelante.

In [13]:
# Se junta el DataFrame en un solo bloque de memoria antes de agregar la columna
df = df.copy()
df["dia_estimado_23h"] = 0
df.loc[DIAS_HASTA_LAS_23, "dia_estimado_23h"] = 1

valores_estimados = pd.DataFrame(estimados).reindex(columns=df.columns)
vacias_antes = df.loc[DIAS_HASTA_LAS_23].isna().sum(axis=1)
df.loc[DIAS_HASTA_LAS_23] = df.loc[DIAS_HASTA_LAS_23].fillna(valores_estimados)

pd.DataFrame({
    "frio_23hs": foto(ee, H_23, "frio_kw").reindex(DIAS_HASTA_LAS_23),
    "frio_kw_estimado": df.loc[DIAS_HASTA_LAS_23, "frio_kw"],
    "columnas_vacias_antes": vacias_antes,
    "columnas_vacias_despues": df.loc[DIAS_HASTA_LAS_23].isna().sum(axis=1),
})

,frio_23hs,frio_kw_estimado,columnas_vacias_antes,columnas_vacias_despues
2021-07-13,33704.0,35527.755556,292,6
2022-03-02,19294.0,20336.005556,292,7
2023-02-28,15279.0,16039.444444,286,3


## 3. Totales imposibles

Buscamos los valores "del orden de los millones" que marca la consigna. Un umbral fijo por sí solo no alcanza, porque las columnas miden cosas muy distintas: hay columnas cuyo valor típico es de unidades y otras cuyo valor típico es de cientos de miles. Un millón puede ser un día alto en una columna y un error evidente en otra.

Por eso consideramos imposible un valor solo si cumple **las dos** condiciones:

- Su valor absoluto es de al menos 1.000.000.
- Es más de 5 veces la **mediana** de su columna. Usamos la mediana y no el promedio porque no se deja arrastrar por los extremos que queremos detectar.

Por ejemplo:

- Mediana 500.000 y valor 1.500.000: es el triple de un día típico. Es un día alto, pero posible.
- Mediana 200 y valor 5.000.000: es 25.000 veces un día típico. Es un error de registro.

Las columnas con mediana 0 (equipos que casi nunca funcionan) quedan cubiertas por la misma regla: cualquier valor de un millón o más se marca como imposible.

La tabla de abajo muestra, para cada columna, `max_sobre_mediana` (cuántas veces el máximo supera a la mediana) y la cantidad de negativos, que por ahora solo contamos.

In [14]:
datos = df.drop(columns=["dia_vacio", "dia_estimado_23h", "frio_kw_siguiente"], errors="ignore")

resumen_valores = pd.DataFrame({
    "mediana": datos.median(),
    "minimo": datos.min(),
    "maximo": datos.max(),
    "n_negativos": (datos < 0).sum(),
    "n_mayor_millon": (datos.abs() >= 1e6).sum(),
})
# Cuántas veces el máximo supera al valor típico de la columna
resumen_valores["max_sobre_mediana"] = resumen_valores["maximo"] / resumen_valores["mediana"].replace(0, np.nan)
resumen_valores.round(1)

,mediana,minimo,maximo,n_negativos,n_mayor_millon,max_sobre_mediana
ee_planta_hl,7.3,0.0,1849.2,0,0,254.9
ee_elaboracion_hl,0.8,0.0,134.7,0,0,170.9
ee_bodega_hl,0.7,-0.4,157.0,5,0,235.9
ee_cocina_hl,0.3,0.0,17.5,0,0,57.4
ee_envasado_hl,1.6,0.0,589.1,0,0,361.3
...,...,...,...,...,...,...
m3_tot_gas,16240.4,982.4,23713.1,0,0,1.5
tot_aire_expulsion_totalizadores_aire,933.3,0.0,154752.7,0,0,165.8
tot_fermentacion_cocina,961.1,0.0,196945.5,0,0,204.9
tot_reposo_inferior,5595.8,0.0,11066.5,0,0,2.0


In [15]:
resumen_valores[resumen_valores["n_mayor_millon"] > 0].sort_values("max_sobre_mediana", ascending=False).round(1)

,mediana,minimo,maximo,n_negativos,n_mayor_millon,max_sobre_mediana
kw_mycom_1,227.0,0.0,5540000.0,0,1,24405.3
totalizador_bba_envasado,1407.5,-347332.0,4880770.0,2,1,3467.7
kw_comp_kaeser,2382.0,0.0,4129585.0,0,2,1733.7
kw_linea_2,2828.0,0.0,4495238.0,0,1,1589.5
restos_planta_kw,3570.5,-4492792.3,2368897.1,19,2,663.5
envasado_kw,16408.0,318.0,4504000.0,0,1,274.5
frio_kw,23656.0,2310.0,5572801.0,0,1,235.6
sala_maq_kw,26823.0,2706.0,5575053.0,0,3,207.8
resto_serv_kw,5122.0,-5535737.5,289142.5,41,1,56.5
planta_kw,66851.9,10769.6,2412966.6,0,1,36.1


In [16]:
cols_millon = resumen_valores.index[resumen_valores["n_mayor_millon"] > 0]
extremos = datos[cols_millon].where(datos[cols_millon].abs() >= 1e6).stack().dropna()
extremos.sort_index()

dia                                 
2020-08-10  vapor_de_caldera_4_kg      -9.869770e+06
2021-02-27  totalizador_bba_envasado    4.880770e+06
2021-03-31  frio_kw                     5.572801e+06
            kw_mycom_1                  5.540000e+06
            resto_serv_kw              -5.535738e+06
            sala_maq_kw                 5.575053e+06
2021-04-25  kw_comp_kaeser              3.541221e+06
            sala_maq_kw                 3.570147e+06
2021-10-30  et_envasado_mj              1.019735e+06
            gas_planta_mj               1.471832e+06
2021-11-29  envasado_kw                 4.504000e+06
            kw_linea_2                  4.495238e+06
            restos_planta_kw           -4.492792e+06
2021-12-24  kw_comp_kaeser              4.129585e+06
            sala_maq_kw                 4.208132e+06
2021-12-29  kw_gral_planta              2.413859e+06
            planta_kw                   2.412967e+06
            restos_planta_kw            2.368897e+06
dtype: fl

Estos valores no son días de consumo enorme. Se ven bien en las lecturas horarias: por ejemplo, el 2021-11-29 `envasado_kw` arranca en 0, salta más de 4 millones entre las 05:00 y las 06:00, y después sigue sumando a su ritmo normal. Es una lectura mala, no un consumo real.

In [17]:
ee[(ee["dia"] == "2021-11-29") & ee["t"].between(pd.Timedelta("04:00:00"), pd.Timedelta("08:00:00"))][["dia", "hora", "envasado_kw"]]

,dia,hora,envasado_kw
3697,2021-11-29,04:00:00,1454.0
3698,2021-11-29,05:00:00,1816.0
3699,2021-11-29,06:00:00,4497239.0
3700,2021-11-29,07:00:00,4497585.0
3701,2021-11-29,08:00:00,4497964.0


In [18]:
UMBRAL_MILLON = 1e6
K = 5

mediana = datos.median()
es_imposible = (datos.abs() >= UMBRAL_MILLON) & (datos.abs() > K * mediana.abs())

print("Valores imposibles:", es_imposible.sum().sum())
print("Valores de 1 millón o más que se conservan:", int(((datos.abs() >= UMBRAL_MILLON) & ~es_imposible).sum().sum()))
es_imposible.sum()[lambda s: s > 0]

Valores imposibles: 16
Valores de 1 millón o más que se conservan: 2


planta_kw                   1
envasado_kw                 1
sala_maq_kw                 3
frio_kw                     1
resto_serv_kw               1
restos_planta_kw            2
kw_gral_planta              1
kw_linea_2                  1
kw_mycom_1                  1
kw_comp_kaeser              2
vapor_de_caldera_4_kg       1
totalizador_bba_envasado    1
dtype: int64

In [19]:
# Los KPI por hectolitro se calculan con estas columnas de kW, así que heredan el mismo error
# aunque no lleguen al millón (ee_frio_hl da 712 el 2021-03-31, con una mediana cercana a 2,7)
KPI_DERIVADOS = {
    "frio_kw": "ee_frio_hl",
    "sala_maq_kw": "ee_sala_maq_hl",
    "resto_serv_kw": "ee_resto_serv_hl",
    "envasado_kw": "ee_envasado_hl",
    "restos_planta_kw": "ee_resto_planta_hl",
    "planta_kw": "ee_planta_hl",
}

df[datos.columns] = datos.mask(es_imposible)
for columna_kw, kpi in KPI_DERIVADOS.items():
    df.loc[es_imposible[columna_kw], kpi] = np.nan

print("KPI derivados vaciados:", sum(int(es_imposible[c].sum()) for c in KPI_DERIVADOS))

KPI derivados vaciados: 9


**Decisión: vaciamos estos valores y no los imputamos en esta etapa.** Son 16 valores en 7 días, y además vaciamos los KPI por hectolitro que se calculan con ellos, porque si no el error seguiría entrando al modelo por otra columna.

No alcanzan la regla los dos valores del 2021-10-30 (`gas_planta_mj` y `et_envasado_mj`): superan el millón, pero son apenas unas 3 veces la mediana de su columna, así que los conservamos.

El único valor imposible de `frio_kw` es el del 2021-03-31. Como es la variable que queremos predecir, intentamos recuperarlo.

### Recuperación de `frio_kw` del 2021-03-31

Si este día queda vacío perdemos también el target del 2021-03-30, y es un dato que se puede rescatar: el error es un único salto y el resto del día suma con normalidad.

Las lecturas de este día están en el archivo 2020-2022, apiladas como segundo bloque del 2021-04-30. Es el mismo bloque que el EDA reasignó al 2021-03-31 (ver "Días pisados").

In [20]:
hoja_ee = pd.read_excel(carpeta_datos / "Totalizadores Planta 2020_2022.xlsx", sheet_name="Consolidado EE", engine="calamine")
hoja_ee.columns = snake_case_unicas(list(hoja_ee.columns))
hoja_ee, _ = limpiar_numericas(hoja_ee)
hoja_ee["dia"] = pd.to_datetime(hoja_ee["dia"], errors="coerce").dt.normalize()
hoja_ee["t"] = pd.to_timedelta(hoja_ee["hora"].astype(str), errors="coerce").dt.floor("min")
hoja_ee = hoja_ee.dropna(subset=["dia", "t"])
hoja_ee["bloque"] = (hoja_ee.groupby("dia")["t"].diff() < pd.Timedelta(0)).groupby(hoja_ee["dia"]).cumsum()

lecturas = hoja_ee[(hoja_ee["dia"] == "2021-04-30") & (hoja_ee["bloque"] == 1)]
assert lecturas["frio_kw"].iloc[-1] == datos.loc["2021-03-31", "frio_kw"]

lecturas[["hora", "frio_kw"]].assign(incremento=lecturas["frio_kw"].diff()).head(5)

,hora,frio_kw,incremento
7573,00:00:00,0.0,NaN
7574,01:00:00,5541542.0,5541542.0
7575,02:00:00,5542848.0,1306.0
7576,03:00:00,5544353.0,1505.0
7577,04:00:00,5545978.0,1625.0


El salto está en el primer incremento, de las 00:00 a la 01:00. Para reconstruir el total sacamos ese incremento y, en su lugar, sumamos la mediana de los incrementos de las demás horas, porque esa primera hora también tuvo consumo real.

Probamos el método en los días sanos: quitamos su primer incremento, lo reemplazamos de la misma forma y comparamos el resultado con el total real.

In [21]:
def total_sin_primer_incremento(valores):
    incrementos = np.diff(valores)
    return valores[-1] - incrementos[0] + np.median(incrementos[1:])

hoja_ee_primer_bloque = hoja_ee[hoja_ee["bloque"] == 0].drop(columns="bloque")
lecturas_validas = pd.concat([hoja_ee_primer_bloque, ee[~ee["dia"].isin(hoja_ee_primer_bloque["dia"])]])

filas = []
for dia, g in lecturas_validas.groupby("dia"):
    valores = g["frio_kw"].to_numpy(dtype=float)
    dia_sano = (
        len(valores) == 25 and g["t"].iloc[-1] == H_2359 and not np.isnan(valores).any()
        and valores[0] == 0 and valores[-1] < 1e6 and (np.diff(valores) >= 0).all()
    )
    if dia_sano:
        filas.append({"dia": dia, "real": valores[-1], "estimado": total_sin_primer_incremento(valores)})

prueba = pd.DataFrame(filas).set_index("dia")
error = prueba["estimado"] - prueba["real"]
print("Días usados en la prueba:", len(prueba))
print("MAE:", round(error.abs().mean(), 1), "kW | Sesgo:", round(error.mean(), 1), "kW | Error p95:", round(error.abs().quantile(0.95), 1), "kW")

Días usados en la prueba: 918
MAE: 194.6 kW | Sesgo: 46.5 kW | Error p95: 509.3 kW


In [22]:
frio_recuperado = total_sin_primer_incremento(lecturas["frio_kw"].to_numpy(dtype=float))

# Se junta el DataFrame en un solo bloque de memoria antes de agregar la columna
df = df.copy()
df["frio_reconstruido"] = 0
df.loc["2021-03-31", "frio_reconstruido"] = 1
df.loc["2021-03-31", "frio_kw"] = frio_recuperado

# El target se recalcula recién ahora, con frio_kw ya limpio
df["frio_kw_siguiente"] = df["frio_kw"].shift(-1)

print("frio_kw recuperado del 2021-03-31:", frio_recuperado)
print("Máximo de frio_kw después de limpiar:", df["frio_kw"].max())
print("Días sin frio_kw:", df["frio_kw"].isna().sum())
print("Target del 2021-03-30:", df.loc["2021-03-30", "frio_kw_siguiente"])
df.loc["2021-03-28":"2021-04-03", ["frio_kw", "frio_reconstruido"]]

frio_kw recuperado del 2021-03-31: 32654.0
Máximo de frio_kw después de limpiar: 56128.0
Días sin frio_kw: 3
Target del 2021-03-30: 32654.0


,frio_kw,frio_reconstruido
dia,,
2021-03-28,33382.0,0
2021-03-29,30952.0,0
2021-03-30,29214.0,0
2021-03-31,32654.0,1
2021-04-01,27220.0,0
2021-04-02,29945.0,0
2021-04-03,22380.0,0


**Decisión: recuperamos este valor y lo marcamos en `frio_reconstruido`.** El resultado queda en línea con los días vecinos y el error de la prueba está muy por debajo del límite de 3.000 kW de la consigna.

Solo aplicamos esto a `frio_kw`. Las demás columnas siguen vacías porque el método no sirve para todas: `sala_maq_kw` tiene dos saltos el 2021-04-25, y `kw_mycom_1` daría 0 el 2021-03-31. `ee_frio_hl` de ese día también queda vacío.

## 4. Días sin reinicio del totalizador

Esto ya se analizó en el EDA (`notebooks/eda_1_1.ipynb`, sección "Días en los que no se reinició el contador", con la función `acumuladores_sin_reiniciar` de `src/eda_1_1_auxiliar.py`). Se buscaron los días cuyo primer registro no es nulo y cuyas lecturas son estrictamente crecientes.

La conclusión fue que **no hay casos que requieran cambios**: los que aparecen son columnas que no deben reiniciarse (niveles y temperaturas, como `nivel_silo_bagazo_norte_1`), diferencias chicas, o diferencias que no se explican por un acumulador sin reiniciar. Además, usamos el valor de las 23:59, así que las columnas que se reinician un poco después de las 00:00 tampoco nos afectan.

Por eso, en este notebook no aplicamos ninguna corrección por este motivo.